In [1]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Assessing How Much Data will be Missing after Concatenating Observed Job Exposure Data (March 2026) and Anthropic Economic Index Data (August 2026) to MCA Job Titles

The goal of this analysis is to determine how many MCA occupations would be lost due to missing Observed Exposure and Automation and Augmentation values. Of the 1,151 occupations, 1,060 (92%) have Observed Exposure data, 907 (79%) have Anthropic Economic Index data, and 852 (74%) have both.

Since the missing values are determined by whether the SOC code is represented in each dataset, mapping in either direction results in the same coverage constraint. Although the drop-off from Anthropic Economic Index data to occupations with both measures is relatively small (79% to 74%), recalculating job exposure using the newer Anthropic data is preferable since it provides more up-to-date measures while allowing us to retain more occupations rather than relying on requiring a job to have both the Observed Exposure data and Anthropic Economic Index data. 

In [2]:
# have this list just to see the funnel of how many codes have some value for a metric
num_jobs_after_merge = []

In [3]:
# load up first the data of mca and get its set of SOC codes
mca_df = pd.read_csv('../data/auxiliary/final_mca_soc_code.csv')
mca_codes = set(mca_df['SOC Code'])
num_jobs_after_merge.append(mca_df.shape[0])

# get the data from last march that has the observed exposure
occupations_observed_exposure_df = pd.read_csv('../data/ai_measurements/job_exposure.csv')
occupations_observed_exposure_df['Code'] = occupations_observed_exposure_df['occ_code'].astype(str) + '.00'
occupations_observed_exposure_df.index = occupations_observed_exposure_df['Code']
occupations_observed_exposure_df.index.name = 'O*NET-SOC Code'
occupations_observed_exposure_df = occupations_observed_exposure_df[['observed_exposure']]

occupations_observed_exposure_codes = set(occupations_observed_exposure_df.index)
codes_obs_exposure_dict = occupations_observed_exposure_df['observed_exposure'].to_dict()

In [4]:
# map SOC codes to observed exposure
mca_df["Observed Exposure"] = mca_df["SOC Code"].map(codes_obs_exposure_dict)

num_mca_codes = len(mca_df)
num_observed_exposure = mca_df["Observed Exposure"].notna().sum()

print(
    f"Out of the {num_mca_codes} jobs in the MCA, "
    f"{num_observed_exposure} ({num_observed_exposure / num_mca_codes:.0%}) "
    "have a corresponding Observed Exposure value in the March 2026 data."
)

num_jobs_after_merge.append(int(num_observed_exposure))

Out of the 1151 jobs in the MCA, 1060 (92%) have a corresponding Observed Exposure value in the March 2026 data.


In [5]:
# get the anthropic data that has automation v augmentation
anthropic_df = pd.read_csv('../data/ai_measurements/release_2026_06_26/data/aei_claude_ai_2026-06-26.csv')
date_end = "2026-05-01"
query = (
    "geo_id == 'GLOBAL' and "
    "category_name == 'soc_occupation' and "
    "hierarchy_level == 0 and "
    f"date_end == '{date_end}'"
)
anthropic_global_df = anthropic_df.query(query)
soc_values_df = (
    anthropic_global_df
    .pivot(
        index=['node_name', 'node_external_id'], 
        columns='metric_id', 
        values='value'
    ).reset_index()
)
# set the index as SOC Code
soc_values_df.index = soc_values_df['node_external_id']
soc_values_df.index.name = 'O*NET-SOC Code'

# get only the relevant columns
automation_augmentation_cols = [
 'collaboration_bucket_augmentation_pct',
 'collaboration_bucket_automation_pct',
 'collaboration_directive_pct',
 'collaboration_feedback_loop_pct',
 'collaboration_learning_pct',
 'collaboration_none_pct',
 'collaboration_task_iteration_pct',
 'collaboration_validation_pct'
]
soc_values_df = soc_values_df[automation_augmentation_cols]

In [6]:
# merge Anthropic data with MCA
mca_df = mca_df.merge(
    soc_values_df.reset_index(),
    how="left",
    left_on="SOC Code",
    right_on="O*NET-SOC Code",
)


# count MCA codes with Anthropic data
anthropic_cols = soc_values_df.columns.difference(
    ["node_name", "node_external_id"]
)

has_anthropic_data = mca_df[anthropic_cols].notna().all(axis=1)
num_anthropic_codes = has_anthropic_data.sum()

print(
    f"Out of the {num_mca_codes} jobs in the MCA, "
    f"{num_anthropic_codes} ({num_anthropic_codes / num_mca_codes:.0%}) "
    "have corresponding data in the June 2026 Anthropic Economic Index."
)

Out of the 1151 jobs in the MCA, 907 (79%) have corresponding data in the June 2026 Anthropic Economic Index.


In [7]:
# count MCA codes with both observed exposure and Anthropic data
has_observed_exposure = mca_df["Observed Exposure"].notna()
has_both = has_observed_exposure & has_anthropic_data

num_codes_after_merge = has_both.sum()

print(
    f"Out of the {num_mca_codes} jobs in the MCA, "
    f"{num_codes_after_merge} ({num_codes_after_merge / num_mca_codes:.0%}) "
    "have both Observed Exposure and Anthropic Economic Index data."
)

num_jobs_after_merge.append(int(num_codes_after_merge))

Out of the 1151 jobs in the MCA, 852 (74%) have both Observed Exposure and Anthropic Economic Index data.


# 2. Consolidate the data to have AIOE, Observed Exposure, Augmentation versus Automation Percentage

We will consolidate the data over the SOC codes.

In [8]:
worldbank_df = pd.read_csv(
    '../data/ai_measurements/worldbank_metrics/output/soc_aioe_comple.csv', 
    index_col='O*NET-SOC Code'
)

In [9]:
consolidated_ai_metrics_df = pd.concat(
    [worldbank_df, occupations_observed_exposure_df, soc_values_df], 
    axis=1
)
formal_column_names = [
    'Title',
    'AIOE',
    'Standardized AIOE',
    'Complementarity',
    'C-AIOE',
    'Observed Exposure',
    'Augmentation Percent',
    'Automation Percent', 
    'Directive Percent',
    'Feedback Loop Percent',
    'Learning Percent',
    'None Percent',
    'Task Iteration Percent',
    'Validation Percent',
]
consolidated_ai_metrics_df.columns = formal_column_names

# were adding to remember what release version of Anthropic data we are in
consolidated_ai_metrics_df['Date Data Released'] = date_end
consolidated_ai_metrics_df.to_csv('../data/auxiliary/soc_consolidated_ai_metrics.csv')

In [11]:
consolidated_ai_metrics_df.isnull().sum()

Title                       0
AIOE                       19
Standardized AIOE          19
Complementarity            19
C-AIOE                     19
Observed Exposure         260
Augmentation Percent      321
Automation Percent        321
Directive Percent         320
Feedback Loop Percent     320
Learning Percent          320
None Percent              320
Task Iteration Percent    320
Validation Percent        320
Date Data Released          0
dtype: int64

In [16]:
consolidated_ai_metrics_df[consolidated_ai_metrics_df['Observed Exposure'] > 0].describe()

,AIOE,Standardized AIOE,Complementarity,C-AIOE,Observed Exposure,Augmentation Percent,Automation Percent,Directive Percent,Feedback Loop Percent,Learning Percent,None Percent,Task Iteration Percent,Validation Percent
count,345.000000,345.000000,345.000000,345.000000,345.000000,317.000000,317.000000,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000
mean,6.330312,0.537813,60.583078,4.896224,0.168679,50.375110,49.624921,34.725975,13.495472,17.121572,2.696509,28.415503,3.545094
std,0.353995,0.790886,7.709256,0.595723,0.153004,16.419539,16.419581,14.155098,11.246709,16.127917,5.585609,14.976669,4.346822
min,5.090957,-2.231124,42.312500,3.378591,0.002200,5.160000,18.030000,11.030000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,6.098878,0.020747,54.844136,4.503323,0.046600,41.230000,37.280000,24.317500,5.852500,3.987500,0.610000,17.200000,1.087500
50%,6.482714,0.878306,60.625000,4.931678,0.109600,53.090000,46.910000,31.870000,10.270000,11.100000,1.110000,27.165000,2.480000
75%,6.611129,1.165206,65.541667,5.306995,0.249000,62.720000,58.770000,41.162500,17.220000,27.967500,1.960000,37.727500,4.052500
max,6.754509,1.485544,80.041667,6.270814,0.745100,81.970000,94.840000,81.720000,69.480000,70.920000,47.060000,67.730000,30.580000


In [15]:
consolidated_ai_metrics_df.describe()

,AIOE,Standardized AIOE,Complementarity,C-AIOE,Observed Exposure,Augmentation Percent,Automation Percent,Directive Percent,Feedback Loop Percent,Learning Percent,None Percent,Task Iteration Percent,Validation Percent
count,997.000000,9.970000e+02,997.000000,997.000000,756.000000,695.000000,695.000000,696.000000,696.000000,696.000000,696.000000,696.000000,696.000000
mean,6.089591,-5.131302e-16,61.642507,4.643003,0.076977,48.346791,51.653309,34.706868,15.144224,16.483534,3.217945,26.969612,3.477802
std,0.447592,1.000000e+00,7.887205,0.604749,0.133172,16.743162,16.743246,15.400859,12.400305,15.779666,7.437701,15.300114,4.490109
min,4.827867,-2.818913e+00,37.854167,3.204920,0.000000,2.290000,18.030000,3.590000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,5.683943,-9.062893e-01,55.791667,4.184174,0.000000,37.685000,39.750000,23.830000,6.520000,3.952500,0.520000,14.440000,0.967500
50%,6.119407,6.661371e-02,61.520833,4.638473,0.000000,51.200000,48.800000,32.340000,11.580000,11.095000,1.140000,26.075000,2.325000
75%,6.527678,9.787619e-01,66.854167,5.070149,0.095675,60.250000,62.315000,42.510000,19.210000,25.612500,2.262500,37.422500,4.082500
max,6.756072,1.489036e+00,81.979167,6.270814,0.745100,81.970000,97.710000,92.100000,85.900000,70.920000,82.440000,73.330000,40.000000
